# 03. Demonstração do modelo do grupo (desenvolvimento por CV interna)

Antes de executar este notebook, implemente `build_group_model` e `search_space` em `src/models/group_model.py` (a tabela no início do arquivo indica o ponto de entrada de cada modelo tema) e instale o extra do modelo (ver README). O espaço de busca e o wrapper devem ser justificados a priori (artigo original e documentação), não ajustados a partir de resultados.

O notebook avalia o modelo com a configuração padrão por CV interna, executa uma busca curta de hiperparâmetros e mostra um estudo de sensibilidade com `cv_grid`, tudo no treino do fold externo `OUTER_FOLD` (0, como no enunciado; o estudo pode ser repetido no treino dos demais folds externos, nunca no teste).

**Teste selado.** Este notebook pertence à fase de desenvolvimento: usa somente o TREINO de um fold externo oficial (`get_train_indices`) e mede o desempenho por validação cruzada INTERNA. O teste dos folds externos só é usado uma vez, em `python -m src.pipeline.final_test`, depois do congelamento (`python -m src.pipeline.freeze`).

**Validação é diagnóstico.** O treino de um fold externo contém o teste dos outros dois folds externos. Por isso, os resultados desta fase (`results/validation.csv`, estudos e curvas) não são base de decisões: os sistemas, o controle `group_tuned`, o wrapper, a configuração padrão, o espaço de busca e o orçamento são fixados a priori. Depois de `run_validation`, só são permitidas correções de falha de execução (status `error`, contrato do estimador), registradas no relatório; mudanças motivadas por desempenho não são permitidas.

**Ordem.** Antes de executar este notebook, comite `src/models/group_model.py` (configuração padrão e espaço de busca) e `docs/declaracoes_a_priori.md` (subconjunto de datasets dos estudos, grades e sentido de simplicidade de cada hiperparâmetro). Este notebook é demonstrativo, só deve ser executado depois desse commit e não deve motivar mudanças no espaço de busca.

In [1]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import pandas as pd

from data.load_tabarena import get_train_indices, load_task, read_selection
from src.models.group_model import build_group_model, search_space
from src.pipeline.cv_grid import cv_grid, inner_cv_splits, stratified_subsample
from src.pipeline.evaluate import fit_predict_evaluate
from src.pipeline.tune import tune

OUTER_FOLD = 0   # fold externo cujo TREINO é usado nos estudos (enunciado: fold externo 0)

In [2]:
STUDY_TASKS = {
    'small': 363621,   # blood-transfusion-service-center
    'medium': 363704,  # students_dropout_and_academic_success
    'large': 363628,   # customer_satisfaction_in_airline
}
assert set(STUDY_TASKS.values()).issubset(read_selection())

# Demonstração inicial no representante pequeno; estudos usam o subconjunto declarado.
task_id = STUDY_TASKS['small']
ds = load_task(task_id)
train_idx = get_train_indices(task_id, folds=[OUTER_FOLD])[0]   # somente o treino do fold externo
X_train, y_train = ds.X.iloc[train_idx], ds.y[train_idx]
X_dev, y_dev = stratified_subsample(X_train, y_train, 10_000, seed=42)
splits = inner_cv_splits(y_dev, n_folds=3, seed=42)

## Configuração padrão (CV interna)

In [3]:
rows = []
for inner, (tr, va) in enumerate(splits):
    r = fit_predict_evaluate(build_group_model(seed=42), X_dev.iloc[tr], y_dev[tr],
                             X_dev.iloc[va], y_dev[va])
    rows.append({'inner_fold': inner, **r.to_dict()})
pd.DataFrame(rows)[['inner_fold', 'auc_ovo', 'accuracy', 'cross_entropy', 'total_time_s']]

,inner_fold,auc_ovo,accuracy,cross_entropy,total_time_s
0,0,0.707551,0.771084,0.510037,5.838052
1,1,0.757218,0.777108,0.462247,5.559508
2,2,0.797917,0.807229,0.444055,5.473515


## Busca de hiperparâmetros no treino (orçamento reduzido para demonstração)

In [4]:
tuning = tune(build_group_model, search_space, X_train, y_train, seed=42, n_trials=5, timeout_s=300)
print('Melhores hiperparâmetros:', tuning.best_params)
print(f'AUC OvO na CV interna (otimista: usada na seleção): {tuning.best_score:.4f}')
print(f'Tempo de busca: {tuning.tune_time_s:.1f}s')

Melhores hiperparâmetros: {'norm_methods': ['power'], 'outlier_threshold': 2.7276805754484785, 'softmax_temperature': 1.4180889157778485, 'average_logits': True}
AUC OvO na CV interna (otimista: usada na seleção): 0.7561
Tempo de busca: 92.3s


In [5]:
# Histórico da busca (útil para o relatório: convergência e sensibilidade).
tuning.study.trials_dataframe().sort_values('value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_average_logits,params_norm_methods,params_outlier_threshold,params_softmax_temperature,state
3,3,0.756072,2026-11-01 11:38:48.705987,2026-11-01 11:39:06.728184,0 days 00:00:18.022197,True,power,2.727681,1.418089,COMPLETE
0,0,0.754432,2026-11-01 11:37:49.188553,2026-11-01 11:38:12.914056,0 days 00:00:23.725503,True,none+power+robust,3.045981,0.532946,COMPLETE
1,1,0.754296,2026-11-01 11:38:12.914056,2026-11-01 11:38:33.229290,0 days 00:00:20.315234,False,none+power+quantile_rtdl,3.136387,0.611612,COMPLETE
2,2,0.754293,2026-11-01 11:38:33.229290,2026-11-01 11:38:48.705987,0 days 00:00:15.476697,True,none+power+quantile_rtdl,3.782266,0.825222,COMPLETE
4,4,0.753964,2026-11-01 11:39:06.728184,2026-11-01 11:39:21.526680,0 days 00:00:14.798496,True,none+power+quantile_rtdl,4.233119,0.519251,COMPLETE


## Estudo de sensibilidade com `cv_grid`

Escolha um hiperparâmetro aceito por `build_group_model` e uma grade de valores declarada antes de olhar os resultados. `cv_grid` prepara a subamostra e as partições internas (as mesmas da busca) e delega o laço a `validation_curve` (trabalho do grupo, `src/pipeline/overfitting.py`): mede treino e validação em cada fold interno; a diferença entre as duas curvas é a base da análise de overfitting (notebook 05). O estudo é diagnóstico: não altera o espaço de busca nem o lock.

In [7]:
PARAM = "softmax_temperature"    # ex.: 'n_estimators' (nome aceito por build_group_model)
VALUES = [0.5, 0.7, 0.9, 1.2, 1.5]    # ex.: [1, 2, 4, 8]

if PARAM is None:
    print('Defina PARAM e VALUES para executar o estudo de sensibilidade.')
else:
    grid = cv_grid(build_group_model, PARAM, VALUES, X_train, y_train, cv_folds=3, seed=42)
    display(grid.groupby('value', sort=False)[['auc_ovo_train', 'auc_ovo_val']].agg(['mean', 'std']))

auc_ovo_train           auc_ovo_val          
               mean       std        mean       std
value                                              
0.5        0.815220  0.020355    0.754161  0.045361
0.7        0.815270  0.020278    0.754161  0.045361
0.9        0.815278  0.020290    0.754228  0.045257
1.2        0.814994  0.020728    0.754061  0.045196
1.5        0.815036  0.020639    0.754027  0.045348